In [1]:
import pandas as pd
import os

portfolio_config = {
    'data/holdings/CNDX.csv': 0.15,      # NASDAQ 100
    'data/holdings/IGLN.csv': 0.05,      # Złoto
    'data/holdings/mWIG40TR.csv': 0.04,  # mWIG40
    'data/holdings/URNU.csv': 0.05,      # Uran
    'data/holdings/VFEA.csv': 0.10,      # Rynki wschodzące
    'data/holdings/VVSM.csv': 0.05,      # Półprzewodniki
    'data/holdings/WIG20TR.csv': 0.06,   # WIG20
    'data/holdings/WEBN.csv': 0.40,      # Globalny
    'data/holdings/WSML.csv': 0.10,      # Małe spółki
      
}

dataframes = []

# 2. Pętla wczytująca pliki
for file_path, portfolio_weight in portfolio_config.items():
    try:
        df = pd.read_csv(file_path)
        
        # --- Ekstrakcja nazwy ETF ze ścieżki ---
        # os.path.basename('data/cspx.csv') zwróci samo 'cspx.csv'
        # .replace('.csv', '') wytnie rozszerzenie, a .upper() powiększy litery -> 'CSPX'
        etf_name = os.path.basename(file_path).replace('.csv', '').upper()
        df['Source_ETF'] = etf_name
        
        # Czyszczenie danych (wielkość liter i puste wartości)
        if 'Name' in df.columns:
            df['Name'] = df['Name'].astype(str).str.upper()
            
        cols_to_fill = ['Sector', 'Country', 'Currency']
        for col in cols_to_fill:
            if col in df.columns:
                df[col] = df[col].fillna('UNKNOWN')
                
        # Obliczenie efektywnej wagi
        df['Effective_Weight'] = df['Weight'] * portfolio_weight
        
        dataframes.append(df)
        
    except FileNotFoundError:
        print(f"Błąd: Nie znaleziono pliku {file_path}. Upewnij się, że folder 'data' istnieje.")

# 3. Połączenie wszystkich danych
df_holdings = pd.concat(dataframes, ignore_index=True)

# 4. TOP 100 Pozycji z łączeniem nazw ETF
# Używamy słownika agregacji: dla wagi robimy sumę, dla ETF łączymy unikalne wartości
top_100_companies = df_holdings.groupby(['Name']).agg({ #TODO: 'ISIN code', 
    'Effective_Weight': 'sum',
    'Source_ETF': lambda x: ', '.join(sorted(x.unique()))
}).reset_index()

# Sortowanie i przycięcie do TOP 100
top_100_companies = top_100_companies.sort_values(by='Effective_Weight', ascending=False).head(100)

# 5. Pozostałe agregacje (Sektor, Kraj, Waluta)
sector_alloc = df_holdings.groupby('Sector')['Effective_Weight'].sum().reset_index().sort_values(by='Effective_Weight', ascending=False)
country_alloc = df_holdings.groupby('Country')['Effective_Weight'].sum().reset_index().sort_values(by='Effective_Weight', ascending=False)
currency_alloc = df_holdings.groupby('Currency')['Effective_Weight'].sum().reset_index().sort_values(by='Effective_Weight', ascending=False)

# 6. Funkcja formatująca wyniki do wyświetlenia
def format_to_percentages(df_to_format):
    df_copy = df_to_format.copy()
    df_copy['Allocation_%'] = (df_copy['Effective_Weight'] * 100).round(2).astype(str) + '%'
    return df_copy.drop(columns=['Effective_Weight']).reset_index(drop=True)

# Wyświetlanie wyników
print("🏆 TOP 100 POZYCJI W PORTFELU (W TYM ŹRÓDŁO ETF)")
display(format_to_percentages(top_100_companies))

print("\n🏭 ALOKACJA SEKTOROWA")
display(format_to_percentages(sector_alloc))

print("\n🌍 ALOKACJA GEOGRAFICZNA (KRAJ)")
display(format_to_percentages(country_alloc))

print("\n💵 ALOKACJA WALUTOWA")
display(format_to_percentages(currency_alloc))

🏆 TOP 100 POZYCJI W PORTFELU (W TYM ŹRÓDŁO ETF)


,Name,Source_ETF,Allocation_%
0,GOLD,IGLN,5.0%
1,NVIDIA CORP,"CNDX, VVSM, WEBN",3.7%
2,APPLE INC,"CNDX, WEBN",2.64%
3,MICROSOFT CORP,"CNDX, WEBN",2.14%
4,BROADCOM INC,"CNDX, VVSM, WEBN",1.81%
...,...,...,...
95,CHEVRON CORP,WEBN,0.14%
96,DEVELIA SA,MWIG40TR,0.14%
97,HOME DEPOT INC,WEBN,0.14%
98,BANK OF AMERICA CORP,WEBN,0.14%



🏭 ALOKACJA SEKTOROWA


,Sector,Allocation_%
0,Information Technology,28.85%
1,Financials,11.62%
2,Industrials,9.3%
3,Consumer Discretionary,8.5%
4,Communication Services,6.85%
5,Energy,6.55%
6,Health Care,5.39%
7,Commodities,5.0%
8,Consumer Staples,3.92%
9,Materials,3.61%



🌍 ALOKACJA GEOGRAFICZNA (KRAJ)


,Country,Allocation_%
0,United States,50.52%
1,Poland,10.03%
2,Global,5.0%
3,Canada,3.7%
4,Japan,3.67%
...,...,...
78,Czech Republic,0.01%
79,Egypt,0.0%
80,-,0.0%
81,GB,0.0%



💵 ALOKACJA WALUTOWA


,Currency,Allocation_%
0,USD,56.98%
1,PLN,10.03%
2,UNKNOWN,9.84%
3,EUR,3.99%
4,JPY,3.67%
5,CAD,3.57%
6,GBP,1.8%
7,KRW,1.7%
8,AUD,1.57%
9,HKD,1.26%
